# **Norwegian News Articles**
Project for TDT4310

By: Malin Haugland Høli

## Evaluating topics

### *Imports*

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

c:\NTNU\V26\TDT4310\Project\exploring-norwegian-news-sources\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### *Load dataset*

In [ ]:
df = pd.read_parquet("../combined-2019.parquet")
nob_df = df[df["language"] == "Bokmål"]

exclude_samples = pd.read_parquet('../topic_modeling_bert_sample.parquet')
exclude_samples_index = nob_df.sample(50000, random_state=42).index

# sample the same 50000, then set the index of the parquet samples
exclude_samples.index = exclude_samples_index

df_eval = nob_df[~nob_df.index.isin(exclude_samples.index)].sample(15000, random_state=42)
texts_eval = df_eval["text"].tolist()

### *Load embedding model*

In [7]:
print("Loading embedding model...")
model_name = "NbAiLab/nb-sbert-base"
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)
print("Embedding model loaded.")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10874.28it/s]
BertModel LOAD REPORT from: NbAiLab/nb-sbert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


In [8]:
embeddings_eval = embedding_model.encode(texts_eval, show_progress_bar=True)

Batches: 100%|██████████| 469/469 [23:12<00:00,  2.97s/it]


### *Print top 10 topics for the simple and advanced model*

In [9]:
simple_model_dir = Path("../bertopic_models/NbAiLab_nb-sbert-base/NbAiLab_nb-sbert-base_n50000_default_20260416_161124").resolve()
advanced_model_dir = Path("../bertopic_models/NbAiLab_nb-sbert-base/NbAiLab_nb-sbert-base_ngram1-3_n50000_min_df10_20260409_201329").resolve()

simple_topic_model = BERTopic.load(str(simple_model_dir), embedding_model=embedding_model)
advanced_topic_model = BERTopic.load(str(advanced_model_dir), embedding_model=embedding_model)

In [ ]:
simple_topic_info = simple_topic_model.get_topic_info()
simple_topic_info = simple_topic_info[simple_topic_info.Topic != -1]
advanced_topic_info = advanced_topic_model.get_topic_info()
advanced_topic_info = advanced_topic_info[advanced_topic_info.Topic != -1]

def print_top_topics(topic_model, topic_info, model_name):
    print(f"Top 10 topics for {model_name}:")
    for _, row in topic_info.head(10).iterrows():
        topic_id = row["Topic"]
        keywords = topic_model.get_topic(topic_id)
        keywords_with_scores = [f"{kw} ({score:.3f})" for kw, score in keywords]
        print(f"Topic {topic_id} (Count: {row['Count']}): {keywords_with_scores}")

print_top_topics(simple_topic_model, simple_topic_info, "Simple Model")
print("\n")
print_top_topics(advanced_topic_model, advanced_topic_info, "Advanced Model")

Top 10 topics for Simple Model:
Topic 0 (Count: 7484): ['det (0.030)', 'på (0.026)', 'han (0.025)', 'og (0.025)', 'er (0.023)', 'til (0.021)', 'jeg (0.021)', 'var (0.021)', 'med (0.020)', 'en (0.020)']
Topic 1 (Count: 4497): ['at (0.029)', 'og (0.026)', 'til (0.025)', 'en (0.025)', 'som (0.024)', 'det (0.024)', 'han (0.024)', 'er (0.024)', 'har (0.024)', 'av (0.022)']
Topic 2 (Count: 4436): ['og (0.030)', 'er (0.028)', 'det (0.027)', 'til (0.026)', 'som (0.026)', 'for (0.025)', 'at (0.025)', 'av (0.023)', 'en (0.022)', 'har (0.022)']
Topic 3 (Count: 3061): ['og (0.028)', 'på (0.028)', 'det (0.027)', 'er (0.027)', 'til (0.025)', 'at (0.025)', 'av (0.023)', 'en (0.023)', 'som (0.022)', 'har (0.022)']
Topic 4 (Count: 2391): ['og (0.031)', 'det (0.027)', 'er (0.025)', 'som (0.025)', 'en (0.024)', 'på (0.023)', 'har (0.022)', 'at (0.022)', 'jeg (0.022)', 'hun (0.021)']
Topic 5 (Count: 1983): ['og (0.027)', 'er (0.027)', 'på (0.026)', 'det (0.024)', 'har (0.023)', 'til (0.023)', 'prosent (0.

### *Topic Evaluation*

In [11]:
simple_topics, _ = simple_topic_model.transform(texts_eval, embeddings=embeddings_eval)
advanced_topics, _ = advanced_topic_model.transform(texts_eval, embeddings=embeddings_eval)

2026-04-17 11:38:56,088 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-04-17 11:38:56,206 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


#### *Embedding Compactness*

In [12]:
def embedding_compactness(topics, embeddings):
    scores = []

    for t in set(topics):
        if t == -1:
            continue

        idx = [i for i, x in enumerate(topics) if x == t]

        if len(idx) < 2:
            continue

        embs = embeddings[idx]
        sim = cosine_similarity(embs)

        n = len(idx)
        # calculate average pairwise cosine similarity, excluding self-similarity (diagonal, where sim == 1)
        score = (sim.sum() - n) / (n * (n - 1))
        scores.append(score)

    return np.mean(scores)

#### *Coherence Score*

In [13]:
def coherence_score(model, topics, texts):
    tokenized = [t.split() for t in texts]

    dictionary = Dictionary(tokenized)
    dictionary.filter_extremes(no_below=5, no_above=0.9)

    topic_words = []
    for t in set(topics):
        if t == -1:
            continue
        words = [w for w, _ in model.get_topic(t)[:10]]
        topic_words.append(words)
    
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence="c_v"
    )

    return cm.get_coherence()

#### *Noise Ratio*

In [14]:
def noise_ratio(topics):
    topics = np.array(topics)
    return np.mean(topics == -1)

#### *Evaluate*

In [15]:
def evaluate(model, topics, texts, embeddings, name):
    return {
        "model": name,
        "coherence": coherence_score(model, topics, texts),
        "compactness": embedding_compactness(topics, embeddings),
        "noise_ratio": noise_ratio(topics)
    }

results = []

results.append(
    evaluate(simple_topic_model, simple_topics, texts_eval, embeddings_eval, "simple")
)

results.append(
    evaluate(advanced_topic_model, advanced_topics, texts_eval, embeddings_eval, "advanced")
)
pd.DataFrame(results)

,model,coherence,compactness,noise_ratio
0,simple,0.477468,0.255239,0.034067
1,advanced,0.592393,0.264524,0.037333
